In [1]:
# ============================================================================
# Genentech BioReasoning Challenge — Track A
# Evaluation backbone (final)
#
# Pipeline: blocked validation splits -> prompt -> LLM -> parse -> AUROC scoring
# with error bars. Prompt-only, no tools, no fine-tuning (Track A rules).
#
# Run as a script for the smoke test + baseline:  python track_a_eval.py
# Or import the functions:  from track_a_eval import evaluate_prompt, ...
# ============================================================================

# ---------------------------------------------------------------------------
# 0. Setup
# ---------------------------------------------------------------------------
# !pip install kaggle pandas numpy scikit-learn requests
# !kaggle competitions download -c ml-gen-x-bioreasoning-challenge-track-a -p data

import os
import re
import json
import time

import numpy as np
import pandas as pd
import requests
from sklearn.metrics import roc_auc_score
from concurrent.futures import ThreadPoolExecutor, as_completed



In [49]:
# Config
# ---------------------------------------------------------------------------
API_URL = "https://vurr6lx60hp06e-8000.proxy.runpod.net/v1/chat/completions"
MODEL_NAME = "openai/gpt-oss-120b"

CALL_SEED = 42        # fixed model seed; deterministic at temperature 0
TEMPERATURE = 0.0
MAX_TOKENS = 2048     # headroom for reasoning before the JSON; drop to ~256 if
                      # your prompt forbids prose and emits JSON only (faster)
MAX_WORKERS = 16      # concurrent requests to the endpoint
EVAL_ROWS = 500       # rows per dev split for iteration speed
NEUTRAL = (1 / 3, 1 / 3)

In [3]:
# 1. Load data
# ---------------------------------------------------------------------------
def load_data(data_dir="data"):
    """Unzip if needed, load train/test, assert the disjointness the task guarantees."""
    zip_path = f"{data_dir}/ml-gen-x-bioreasoning-challenge-track-a.zip"
    if os.path.exists(zip_path):
        import zipfile
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(data_dir)
        os.remove(zip_path)

    train = pd.read_csv(f"{data_dir}/train.csv")
    test = pd.read_csv(f"{data_dir}/test.csv")

    assert set(train["pert"]).isdisjoint(set(test["pert"])), "pert overlap!"
    assert set(train["gene"]).isdisjoint(set(test["gene"])), "gene overlap!"
    return train, test

In [4]:
# 2. Scoring metric  (average of DE and DIR micro-AUROCs)
# ---------------------------------------------------------------------------
def score_predictions(df):
    """Needs columns: label ('up'/'down'/'none'), prediction_up, prediction_down."""
    label = df["label"].to_numpy()
    p_up = df["prediction_up"].to_numpy(dtype=float)
    p_down = df["prediction_down"].to_numpy(dtype=float)

    # DE: changed at all (up|down) vs none
    is_de = (label != "none").astype(int)
    de_auroc = roc_auc_score(is_de, p_up + p_down)

    # DIR: among true movers, up vs down
    m = label != "none"
    denom = np.where((p_up + p_down)[m] == 0, 1e-9, (p_up + p_down)[m])
    dir_auroc = roc_auc_score((label[m] == "up").astype(int), p_up[m] / denom)

    return {"de_auroc": de_auroc, "dir_auroc": dir_auroc,
            "score": (de_auroc + dir_auroc) / 2.0}


In [5]:
# 3. Blocked validation splits
#    dev      = pert AND gene both held out  (mimics the real test)
#    develop  = pert AND gene both seen      (safe to draw few-shot examples from)
# ---------------------------------------------------------------------------
def make_blocked_validation_split(train, frac_perts=0.25, frac_genes=0.30, seed=0):
    rng = np.random.default_rng(seed)
    perts, genes = train["pert"].unique(), train["gene"].unique()
    hold_perts = set(rng.choice(perts, int(len(perts) * frac_perts), replace=False))
    hold_genes = set(rng.choice(genes, int(len(genes) * frac_genes), replace=False))
    p_held = train["pert"].isin(hold_perts)
    g_held = train["gene"].isin(hold_genes)
    return train[~p_held & ~g_held].copy(), train[p_held & g_held].copy()


In [6]:
# 4. Robust response parser  (single definition)
#    Returns (p_up, p_down, parsed_ok). Neutral fallback ONLY when unparseable.
# ---------------------------------------------------------------------------
def parse_response(text):
    if not text:
        return NEUTRAL[0], NEUTRAL[1], False
    for blob in reversed(re.findall(r"\{.*?\}", text, flags=re.DOTALL)):
        try:
            obj = json.loads(blob)
        except Exception:
            continue
        up = obj.get("prediction_up", obj.get("p_up"))
        down = obj.get("prediction_down", obj.get("p_down"))
        if up is None or down is None:
            continue
        up = min(max(float(up), 0), 1)
        down = min(max(float(down), 0), 1)
        if up + down > 1:
            up, down = up / (up + down), down / (up + down)
        return up, down, True
    return NEUTRAL[0], NEUTRAL[1], False

In [7]:
# 5. Model call  (single real implementation)
# ---------------------------------------------------------------------------
def call_model(prompt, seed=CALL_SEED, temperature=TEMPERATURE,
               max_tokens=MAX_TOKENS, timeout=120):
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "seed": seed,
        "max_tokens": max_tokens,
    }
    r = requests.post(API_URL, headers={"Content-Type": "application/json"},
                      json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


def call_llm_real(prompt_text, seed=CALL_SEED, temperature=TEMPERATURE,
                  max_tokens=MAX_TOKENS, retries=2):
    """Returns (text, ok). ok=False on failure so callers can track it."""
    for attempt in range(retries + 1):
        try:
            out = call_model(prompt_text, seed=seed, temperature=temperature,
                             max_tokens=max_tokens)
            return out or "", True
        except Exception as e:
            if attempt == retries:
                print("call_llm_real failed:", e)
                return "", False
            time.sleep(1.5 * (attempt + 1))  # backoff for transient 429/timeout


In [10]:
# 6. Run a prompt over a question set  (CONCURRENT, ONE call per row)
#    At temperature 0 the model is deterministic, so one call per row is
#    sufficient — averaging seeds would only triple cost for identical answers.
# ---------------------------------------------------------------------------
def run_prompt(questions, prompt_template, seed=CALL_SEED,
               max_tokens=MAX_TOKENS, max_workers=MAX_WORKERS):
    def one(q):
        pt = prompt_template.format(pert=q["pert"], gene=q["gene"])
        text, ok = call_llm_real(pt, seed=seed, max_tokens=max_tokens)
        pu, pd_, parsed = parse_response(text)
        return {"id": q["id"], "prediction_up": pu, "prediction_down": pd_,
                "call_ok": ok, "parsed": parsed}

    rows = [None] * len(questions)
    recs = questions.to_dict("records")
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(one, q): i for i, q in enumerate(recs)}
        for fut in as_completed(futures):
            rows[futures[fut]] = fut.result()

    df = pd.DataFrame(rows)
    n_fail = int((~df["call_ok"]).sum())
    n_unparsed = int((df["call_ok"] & ~df["parsed"]).sum())
    if n_fail or n_unparsed:
        print(f"  ⚠ {n_fail} call failures, {n_unparsed} unparsed "
              f"(scored as neutral 1/3,1/3 — interpret AUROC with that in mind)")
    return df

def evaluate_prompt(prompt_template, dev_set, seed=CALL_SEED,
                    max_tokens=MAX_TOKENS, max_workers=MAX_WORKERS):
    keep = dev_set[["id", "pert", "gene", "label"]]
    preds = run_prompt(keep, prompt_template, seed=seed,
                       max_tokens=max_tokens, max_workers=max_workers)
    scored = keep.merge(preds, on="id")
    return score_predictions(scored), scored


In [11]:
# 7. Error analysis
# ---------------------------------------------------------------------------
def analyze_errors(scored):
    df = scored.copy()
    df["p_de"] = df["prediction_up"] + df["prediction_down"]
    df["pred"] = np.where(df["p_de"] < 0.5, "none",
                          np.where(df["prediction_up"] >= df["prediction_down"],
                                   "up", "down"))
    print("Confusion (rows=true, cols=pred):")
    print(pd.crosstab(df["label"], df["pred"]))
    print("\nMean P(DE) by true label (want none << up, down):")
    print(df.groupby("label")["p_de"].mean().round(3))
    mv = df[df["label"] != "none"].copy()
    mv["ok"] = (((mv["label"] == "up") & (mv["prediction_up"] >= mv["prediction_down"]))
                | ((mv["label"] == "down") & (mv["prediction_down"] > mv["prediction_up"])))
    print(f"\nDirection accuracy on movers: {mv['ok'].mean():.3f} "
          f"(always-up = {(mv['label'] == 'up').mean():.3f})")


In [12]:
# 8. Multi-split evaluation with error bars
# ---------------------------------------------------------------------------
def evaluate_prompt_multiseed(prompt_template, train, n_seeds=3,
                              n_rows=EVAL_ROWS, seed=CALL_SEED,
                              max_tokens=MAX_TOKENS, max_workers=MAX_WORKERS):
    rows = []
    for sd in range(n_seeds):
        _, dev_sd = make_blocked_validation_split(train, seed=sd)
        if n_rows:
            dev_sd = dev_sd.head(n_rows)
        sc, _ = evaluate_prompt(prompt_template, dev_sd, seed=seed,
                                max_tokens=max_tokens, max_workers=max_workers)
        sc["split_seed"] = sd
        rows.append(sc)
    res = pd.DataFrame(rows)
    summary = {"score_mean": res["score"].mean(), "score_std": res["score"].std(),
               "de_mean": res["de_auroc"].mean(), "de_std": res["de_auroc"].std(),
               "dir_mean": res["dir_auroc"].mean(), "dir_std": res["dir_auroc"].std()}
    return summary, res


In [13]:
# 9. Version log
# ---------------------------------------------------------------------------
version_log = []
def log_version(name, score_dict, notes=""):
    version_log.append({"version": name, **score_dict, "notes": notes})
    return pd.DataFrame(version_log)[["version", "score", "de_auroc", "dir_auroc", "notes"]]


In [14]:
# 10. Self-tests that need no network (run on import — cheap, catch regressions)
# ---------------------------------------------------------------------------
assert parse_response('{"prediction_up":0.6,"prediction_down":0.2}')[:2] == (0.6, 0.2)
assert parse_response('reasoning...\n{"p_up":0.5,"p_down":0.5}')[:2] == (0.5, 0.5)
assert parse_response('garbage')[2] is False


In [45]:
# 11. Script entry point  (smoke test + baseline; only runs as `python ...`)
# ---------------------------------------------------------------------------
prompt_v0 = """You are solving a biological ternary classification task.

Task:
- Input: a perturbation gene (pert) and a target gene (gene).
- Output: probabilities for three labels:
  - up: target gene is upregulated
  - down: target gene is downregulated
  - none: no significant differential expression

Rules:
- Use biological reasoning only.
- Return a probability distribution over the three labels that sums to 1.
- Output ONLY the JSON object below. No markdown, no code fences, no prose.

Output format:
{{"prediction_up": <float>, "prediction_down": <float>, "prediction_none": <float>}}

pert: {pert}
gene: {gene}"""


if __name__ == "__main__":
    train, test = load_data("data")
    print("Train shape:", train.shape, "| Test shape:", test.shape)
    print("Label distribution:\n", train["label"].value_counts())
    print("✓ train/test disjoint on pert and gene")

    # one smoke call to confirm the endpoint is up
    _smoke = ('CRISPRi knockdown of Stat1 in mouse macrophages. Predict the effect on Irf1. '
              'Output ONLY JSON: {"prediction_up": x, "prediction_down": y, "prediction_none": z}')
    _text, _ok = call_llm_real(_smoke)
    print("smoke ok:", _ok, "| parsed:", parse_response(_text)[:2])

    # single split, quick look
    _, dev0 = make_blocked_validation_split(train, seed=0)
    sc, scored = evaluate_prompt(prompt_v0, dev0.head(EVAL_ROWS))
    print("Prompt v0 score:", sc)
    analyze_errors(scored)
    print(log_version("v0", sc, "baseline mechanism prompt"))

    # full error-bar evaluation across 3 splits
    # summary, per_seed = evaluate_prompt_multiseed(prompt_v0, train, n_seeds=3)
    # print(per_seed[["split_seed", "score", "de_auroc", "dir_auroc"]].round(4))
    # print(f"score = {summary['score_mean']:.3f} ± {summary['score_std']:.3f}")

Train shape: (7705, 4) | Test shape: (1813, 3)
Label distribution:
 none    4260
up      2359
down    1086
Name: label, dtype: int64
✓ train/test disjoint on pert and gene
smoke ok: True | parsed: (0.0, 1.0)
  ⚠ 0 call failures, 8 unparsed (scored as neutral 1/3,1/3 — interpret AUROC with that in mind)
Prompt v0 score: {'de_auroc': 0.5942420152946469, 'dir_auroc': 0.6245641224331655, 'score': 0.6094030688639063}
Confusion (rows=true, cols=pred):
pred   down  none  up
label                
down     16    63  10
none     19   217  30
up        5   102  38

Mean P(DE) by true label (want none << up, down):
label
down    0.355
none    0.300
up      0.373
Name: p_de, dtype: float64

Direction accuracy on movers: 0.590 (always-up = 0.620)
  version     score  de_auroc  dir_auroc                      notes
0      v0  0.675162  0.658044   0.692280  baseline mechanism prompt
1      v0  0.609403  0.594242   0.624564  baseline mechanism prompt


In [30]:
prompt_v2 = """You are an expert molecular biologist specializing in gene regulatory networks and CRISPRi Perturb-seq screens.
Experimental setup: a genome-wide CRISPRi knockdown screen in mouse bone marrow-derived macrophages (BMDMs), primary M-CSF-differentiated immune cells. In each case one perturbation gene is knocked down (dCas9 represses its transcription) and the response of a separate target gene is measured by scRNA-seq. An effect is called only at 5% FDR with |log2 fold-change| >= log2(1.5); smaller or noisier changes count as no significant effect.
Task: given a perturbation gene (pert) and a target gene (gene), output a calibrated probability distribution over three labels:
- up:   knockdown of pert UP-regulates gene
- down: knockdown of pert DOWN-regulates gene
- none: no significant differential expression
Reason using these principles:
1. **Baseline “none” prior** – Start with a high probability for “none” because ~55 % of pairs show no effect. Re‑allocate probability to “up”/“down” only when mechanistic evidence justifies it.
2. **Expression & detectability in BMDMs** – If either pert or target is low/absent in mouse bone‑marrow‑derived macrophages, keep “none” dominant. When both are robustly expressed, allow a larger effect weight.
3. **Regulatory class logic** –  
   a. **Transcriptional activators / co‑activators** → knock‑down removes positive drive → downstream targets tend **down**.  
   b. **Transcriptional repressors / negative‑feedback nodes** → removal of inhibition → targets tend **up**.  
   c. **Kinases, phosphatases, adaptors** – follow the sign of the signaling flow; an activating kinase upstream of a repressor produces an **up** effect on the final target, an inhibitory kinase produces **down**.  
   d. **Complex or catalytic subunit** – loss reduces overall complex activity; targets of that activity shift in the same direction as loss of the whole complex.
4. **Pathway propagation with distance weighting** – Trace pert → intermediate → target through known signaling or transcriptional cascades (e.g., NF‑κB, JAK/STAT, MAPK, interferon, lipid metabolism). Apply the sign rules step‑by‑step and multiply the “up + down” weight by a decay factor that decreases with each additional edge (strongest for direct interaction, moderate for 2‑step, weak beyond).
5. **Transcription‑factor cascade effect** – If the perturbed gene encodes a TF (or a regulator of a TF) that directly controls the target’s promoter, assign high confidence to the direction given by the TF’s mode (activator vs repressor). If the link is via a secondary TF whose expression changes after knock‑down, give a moderate confidence boost.
6. **Regulon and co‑expression coupling** – When pert and target share a regulon (common TF binding, same GO biological process, or strong co‑expression in macrophages), increase the combined “up + down” weight. Prioritize: physical interaction > same complex > same pathway > shared process.
7. **Context‑dependent secondary responses** –  
   a. Loss of essential housekeeping, ribosomal, or chromatin‑remodeling genes often triggers stress programs (e.g., unfolded‑protein response, type‑I IFN). If the target belongs to a stress‑responsive set, bias toward **up**.  
   b. Loss of a core activator of a specialized macrophage program (cytokine receptor, metabolic enzyme, differentiation factor) biases toward **down** for genes characteristic of that program.  
   c. Secreted ligands or receptors that modulate activation states can cause indirect up‑regulation of surface‑marker genes; treat such signaling cues as a moderate “up” boost when the target is an activation marker.
8. **Macrophage‑specific functional context** – Give extra weight when both genes are known participants in innate‑immune signaling, cytokine production, or metabolic reprogramming typical of BMDMs; otherwise retain the baseline “none” dominance.
Output rules:
- Output ONLY a single JSON object, no markdown, no code fences, no text before or after.
- The three probabilities must sum to 1.
- Keep "reasoning" to one or two sentences naming the specific mechanism or interaction you relied on (or its absence).
Output format (note: reasoning comes FIRST so you commit to the mechanism before the probabilities):
{
  "reasoning": "<short mechanism-based reasoning>",
  "prediction_up": <float>,
  "prediction_down": <float>,
  "prediction_none": <float>
}
Now solve this case:
pert: <<PERT>>
gene: <<GENE>>
...
Output format:
{
  "reasoning": "<short mechanism-based reasoning>",
  "prediction_up": <float>,
  "prediction_down": <float>,
  "prediction_none": <float>
}
"""   # ← paste the prompt EXACTLY, braces and all, no doubling

In [31]:
def run_prompt_str(questions, prompt_text, seed=CALL_SEED,
                   max_tokens=MAX_TOKENS, max_workers=MAX_WORKERS):
    """Append pert/gene as a suffix — no .format(), so braces need no escaping."""
    def one(q):
        pt = f'{prompt_text}\n\npert: {q["pert"]}\ngene: {q["gene"]}'
        text, ok = call_llm_real(pt, seed=seed, max_tokens=max_tokens)
        pu, pd_, parsed = parse_response(text)
        return {"id": q["id"], "prediction_up": pu, "prediction_down": pd_,
                "call_ok": ok, "parsed": parsed}

    rows = [None] * len(questions)
    recs = questions.to_dict("records")
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(one, q): i for i, q in enumerate(recs)}
        for fut in as_completed(futures):
            rows[futures[fut]] = fut.result()
    df = pd.DataFrame(rows)
    n_fail = int((~df["call_ok"]).sum())
    n_unparsed = int((df["call_ok"] & ~df["parsed"]).sum())
    if n_fail or n_unparsed:
        print(f"  ⚠ {n_fail} call failures, {n_unparsed} unparsed (neutral 1/3,1/3)")
    return df

def evaluate_prompt_str(prompt_text, dev_set, **kw):
    keep = dev_set[["id", "pert", "gene", "label"]]
    preds = run_prompt_str(keep, prompt_text, **kw)
    scored = keep.merge(preds, on="id")
    return score_predictions(scored), scored

In [36]:
sc, scored = evaluate_prompt_str(prompt_v2, dev0.head(EVAL_ROWS))
print(sc)
analyze_errors(scored) 

  ⚠ 0 call failures, 1 unparsed (neutral 1/3,1/3)
{'de_auroc': 0.6090546880020564, 'dir_auroc': 0.651685393258427, 'score': 0.6303700406302417}
Confusion (rows=true, cols=pred):
pred   down  none  up
label                
down      7    82   0
none      5   258   3
up        1   138   6

Mean P(DE) by true label (want none << up, down):
label
down    0.299
none    0.251
up      0.294
Name: p_de, dtype: float64

Direction accuracy on movers: 0.607 (always-up = 0.620)


In [42]:
prompt_v3= """You are an expert molecular biologist specializing in gene regulatory networks and CRISPRi Perturb-seq screens.
Experimental setup: a genome-wide CRISPRi knockdown screen in mouse bone marrow-derived macrophages (BMDMs), primary M-CSF-differentiated immune cells. In each case one perturbation gene is knocked down (dCas9 represses its transcription) and the response of a separate target gene is measured by scRNA-seq. An effect is called only at 5% FDR with |log2 fold-change| >= log2(1.5); smaller or noisier changes count as no significant effect.
Task: given a perturbation gene (pert) and a target gene (gene), output a calibrated probability distribution over three labels:
- up:   knockdown of pert UP-regulates gene
- down: knockdown of pert DOWN-regulates gene
- none: no significant differential expression
Reason using these principles:
1. Base rate. The large majority of (pert, gene) pairs show no significant effect. "none" should usually carry the most mass unless there is a concrete mechanistic link. Do not assign high up/down probability on a weak or speculative connection.
2. Direction logic.
   - If pert normally activates or is required for gene (upstream transcriptional activator, co-activator, or a pathway component that drives gene), knockdown should DOWN-regulate gene.
   - If pert normally represses gene (transcriptional repressor, or driver of a negative-feedback / competing program), knockdown should UP-regulate gene.
   - Indirect effects via shared complexes, pathways, or feedback can flip the naive sign; weigh them.
3. Evidence to weigh: known protein-protein interactions, shared pathways or complexes, transcription-factor -> target relationships, and whether both genes act in macrophage-relevant programs (NF-kB / inflammatory signaling, type I and type II interferon, phagocytosis, lipid handling, metabolic stress). A direct or close functional link raises the chance of a real effect; no plausible link argues for "none".
4. Macrophage context. Prioritize regulatory relationships active in differentiated BMDMs over generic or cell-type-irrelevant ones.
5. Resolution. Use the full 0-1 range and give graded, fine-grained probabilities that track how strong the evidence is; avoid collapsing many different cases onto the same few round numbers, so that more strongly linked pairs receive a higher up+down total than weaker ones.
Output rules:
- Output ONLY a single JSON object, no markdown, no code fences, no text before or after.
- The three probabilities must sum to 1.
- Keep "reasoning" to one or two sentences naming the specific mechanism or interaction you relied on (or its absence).
Output format (note: reasoning comes FIRST so you commit to the mechanism before the probabilities):
{
  "reasoning": "<short mechanism-based reasoning>",
  "prediction_up": <float>,
  "prediction_down": <float>,
  "prediction_none": <float>
}
Few-shot examples:
{
  "pert": "Eif5",
  "gene": "Mtdh",
  "label": "up"
}
{
  "pert": "Gpn2",
  "gene": "Polr2a",
  "label": "up"
}
{
  "pert": "Pi4kb",
  "gene": "Mir22hg",
  "label": "up"
}
{
  "pert": "Snapc3",
  "gene": "Neat1",
  "label": "up"
}
{
  "pert": "Axin1",
  "gene": "Cd36",
  "label": "up"
}
{
  "pert": "Gspt1",
  "gene": "Atf6",
  "label": "up"
}
{
  "pert": "Tgif1",
  "gene": "Ebi3",
  "label": "up"
}
{
  "pert": "Rela",
  "gene": "Clec4e",
  "label": "down"
}
{
  "pert": "Tpt1",
  "gene": "Fgl2",
  "label": "down"
}
{
  "pert": "Sptssa",
  "gene": "Blvrb",
  "label": "down"
}
{
  "pert": "Irak4",
  "gene": "Ifitm2",
  "label": "down"
}
{
  "pert": "Urm1",
  "gene": "Ckb",
  "label": "down"
}
{
  "pert": "Atxn7l3",
  "gene": "Ctsl",
  "label": "down"
}
{
  "pert": "Fzr1",
  "gene": "Pld4",
  "label": "down"
}
{
  "pert": "Smarcd2",
  "gene": "Gm47507",
  "label": "none"
}
{
  "pert": "Rbbp6",
  "gene": "2810001A02Rik",
  "label": "none"
}
{
  "pert": "Rps19",
  "gene": "Btg1",
  "label": "none"
}
{
  "pert": "Nelfa",
  "gene": "Kars",
  "label": "none"
}
{
  "pert": "Cdc123",
  "gene": "Akr1a1",
  "label": "none"
}
{
  "pert": "Rptor",
  "gene": "Bdp1",
  "label": "none"
}
{
  "pert": "Wdr75",
  "gene": "Psd3",
  "label": "none"
}
Now solve this case:
pert: {{PERT}}
gene: {{GENE}}
"""

In [38]:
  def run_prompt_v3(questions, prompt_text, seed=CALL_SEED,
                  max_tokens=MAX_TOKENS, max_workers=MAX_WORKERS):
    """For prompts with {{PERT}}/{{GENE}} placeholders already inside the text."""
    def one(q):
        pt = (prompt_text
              .replace("{{PERT}}", str(q["pert"]))
              .replace("{{GENE}}", str(q["gene"])))
        text, ok = call_llm_real(pt, seed=seed, max_tokens=max_tokens)
        pu, pd_, parsed = parse_response(text)
        return {"id": q["id"], "prediction_up": pu, "prediction_down": pd_,
                "call_ok": ok, "parsed": parsed}

    rows = [None] * len(questions)
    recs = questions.to_dict("records")
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(one, q): i for i, q in enumerate(recs)}
        for fut in as_completed(futures):
            rows[futures[fut]] = fut.result()

    df = pd.DataFrame(rows)
    n_fail = int((~df["call_ok"]).sum())
    n_unparsed = int((df["call_ok"] & ~df["parsed"]).sum())
    if n_fail or n_unparsed:
        print(f"  ⚠ {n_fail} call failures, {n_unparsed} unparsed (neutral 1/3,1/3)")
    return df

def evaluate_prompt_v3(prompt_text, dev_set, **kw):
    keep = dev_set[["id", "pert", "gene", "label"]]
    preds = run_prompt_v3(keep, prompt_text, **kw)
    scored = keep.merge(preds, on="id")
    return score_predictions(scored), scored

In [43]:
q = dev0.iloc[0]
pt = prompt_v3.replace("{{PERT}}", str(q["pert"])).replace("{{GENE}}", str(q["gene"]))
print("{{PERT}} left:", "{{PERT}}" in pt, "| {{GENE}} left:", "{{GENE}}" in pt)
print(pt[-200:])   # confirm the tail shows real pert/gene, not literal {{PERT}}

{{PERT}} left: False | {{GENE}} left: False
gene": "Akr1a1",
  "label": "none"
}
{
  "pert": "Rptor",
  "gene": "Bdp1",
  "label": "none"
}
{
  "pert": "Wdr75",
  "gene": "Psd3",
  "label": "none"
}
Now solve this case:
pert: Psmd4
gene: Anxa2



In [50]:
sc3, scored3 = evaluate_prompt_v3(prompt_v3, dev0.head(EVAL_ROWS))
print(sc3)
analyze_errors(scored3)

{'de_auroc': 0.5895748987854251, 'dir_auroc': 0.6991088725300271, 'score': 0.6443418856577261}
Confusion (rows=true, cols=pred):
pred   down  none  up
label                
down     10    79   0
none      2   261   3
up        0   138   7

Mean P(DE) by true label (want none << up, down):
label
down    0.216
none    0.138
up      0.179
Name: p_de, dtype: float64

Direction accuracy on movers: 0.654 (always-up = 0.620)


In [51]:
# 1. Paste the prompt as a string (the whole thing from your teammates)
prompt_v4 = """You are an expert molecular biologist specializing in gene regulatory networks and CRISPRi Perturb-seq screens.
Experimental setup: a genome-wide CRISPRi knockdown screen in mouse bone marrow-derived macrophages (BMDMs), primary M-CSF-differentiated immune cells. In each case one perturbation gene is knocked down (dCas9 represses its transcription) and the response of a separate target gene is measured by scRNA-seq. An effect is called only at 5% FDR with |log2 fold-change| >= log2(1.5); smaller or noisier changes count as no significant effect.
Task: given a perturbation gene (pert) and a target gene (gene), output a calibrated probability distribution over three labels:
- up:   knockdown of pert UP-regulates gene
- down: knockdown of pert DOWN-regulates gene
- none: no significant differential expression
Reason using these principles:
1. Baseline "none" prior. ~55% of pairs show no effect. Re-allocate probability to up/down only when the mechanism justifies it.
2. Regulatory class logic. Activators/co-activators or required drivers -> knockdown lowers the target (down). Repressors / negative-feedback nodes -> knockdown raises the target (up). For kinases/phosphatases/adaptors/complex subunits, follow the net sign of the signaling flow.
3. Pathway propagation. Trace pert -> intermediate -> target through known cascades (NF-kB, JAK/STAT, MAPK, interferon, lipid/metabolic), applying the sign at each edge, with weight decaying as the chain gets longer.
4. Macrophage context. Up-weight relationships active in differentiated mouse BMDMs.
After writing your reasoning chain, and BEFORE assigning probabilities, AUDIT YOUR OWN CHAIN FOR HALLUCINATION. Empirically, some chain types are reliable and others are usually fabricated:
RELIABLE chains -> you may keep substantial up/down mass and trust the direction:
- A perturbation that impairs ER / secretory / glycosylation / translocon function or protein folding -> ER stress -> UPR -> UP-regulation of UPR targets (Hspa5/BiP, Dnajb9, chaperones). This is the single most reliable pattern.
- A perturbation that impairs the proteasome / ubiquitin-proteostasis system -> proteotoxic stress response.
- A clear relief-of-repression / negative-feedback step (knockdown removes a repressor, so the target rises).
HALLUCINATION-PRONE chains -> pull most of the mass back to "none" and keep up/down small (these are right only ~half the time, i.e. no better than a guess):
- Any SPECIFIC regulatory edge you are asserting from memory: "X directly activates/represses gene Y", "X binds Y's promoter", "X is the TF for Y". You usually do NOT actually know these specific edges; do not trust their direction.
- Generic indirect links via a broad shared pathway or co-membership: "both in NF-kB / JAK-STAT signaling", "same complex", "shared GO process", with no concrete, well-known edge. This is usually post-hoc storytelling.
- A long multi-step chain (3+ hops) assembled just to reach a direction.
Decision rule: assign confident up/down ONLY when your chain is a RELIABLE type above, or a textbook-level direct relationship you are genuinely certain of. If your chain rests on a hallucination-prone structure, keep "none" as the dominant mass.
Output rules:
- Output ONLY a single JSON object, no markdown, no code fences, no text before or after.
- The three probabilities must sum to 1.
- "reasoning" (one or two sentences): state the chain, then your hallucination audit verdict (reliable vs hallucination-prone) and the resulting direction.
Output format (reasoning FIRST so you commit to the mechanism and the audit before the numbers):
{
  "reasoning": "<chain + hallucination-audit verdict + implied direction>",
  "prediction_up": <float>,
  "prediction_down": <float>,
  "prediction_none": <float>
}
Few-shot examples:
{{FEWSHOT}}
Now solve this case:
pert: {{PERT}}
gene: {{GENE}}
"""

# 2. Remove the few-shot placeholder (zero-shot)
prompt_v4 = prompt_v4.replace("{{FEWSHOT}}", "")

In [52]:
sc4, scored4 = evaluate_prompt_v3(prompt_v4, dev0.head(EVAL_ROWS))
print(sc4)
analyze_errors(scored4)
print(log_version("v4", sc4, "audit-for-hallucination prompt, zero-shot"))

{'de_auroc': 0.5418353576248314, 'dir_auroc': 0.6535451375435878, 'score': 0.5976902475842096}
Confusion (rows=true, cols=pred):
pred   down  none  up
label                
down      8    81   0
none      2   262   2
up        0   128  17

Mean P(DE) by true label (want none << up, down):
label
down    0.241
none    0.202
up      0.265
Name: p_de, dtype: float64

Direction accuracy on movers: 0.637 (always-up = 0.620)
  version     score  de_auroc  dir_auroc  \
0      v0  0.675162  0.658044   0.692280   
1      v0  0.609403  0.594242   0.624564   
2      v4  0.597690  0.541835   0.653545   

                                       notes  
0                  baseline mechanism prompt  
1                  baseline mechanism prompt  
2  audit-for-hallucination prompt, zero-shot  


In [53]:
import pandas as pd
import numpy as np

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

print("="*60)
print("DATASET SIZE")
print("="*60)
print(f"Train rows: {len(train):,}")
print(f"Test rows:  {len(test):,}")
print(f"Columns:    {list(train.columns)}")

print("\n" + "="*60)
print("LABEL DISTRIBUTION (the class balance — key slide fact)")
print("="*60)
counts = train["label"].value_counts()
pct = train["label"].value_counts(normalize=True).mul(100).round(1)
for lab in ["up", "down", "none"]:
    print(f"  {lab:5s}: {counts.get(lab,0):>6,}  ({pct.get(lab,0):>5.1f}%)")
movers = (train["label"] != "none").sum()
print(f"  movers (up+down): {movers:,} ({movers/len(train)*100:.1f}%)")

print("\n" + "="*60)
print("UNIQUE ENTITIES")
print("="*60)
print(f"  Unique perts (train): {train['pert'].nunique():,}")
print(f"  Unique genes (train): {train['gene'].nunique():,}")
print(f"  Unique perts (test):  {test['pert'].nunique():,}")
print(f"  Unique genes (test):  {test['gene'].nunique():,}")

print("\n" + "="*60)
print("TRAIN/TEST DISJOINTNESS (why blocked validation is needed)")
print("="*60)
pert_overlap = len(set(train['pert']) & set(test['pert']))
gene_overlap = len(set(train['gene']) & set(test['gene']))
print(f"  Shared perts: {pert_overlap}  (0 = fully disjoint)")
print(f"  Shared genes: {gene_overlap}  (0 = fully disjoint)")

print("\n" + "="*60)
print("PAIR COVERAGE (how sparse the pert×gene matrix is)")
print("="*60)
possible = train['pert'].nunique() * train['gene'].nunique()
print(f"  Observed pairs:        {len(train):,}")
print(f"  Possible pert×gene:    {possible:,}")
print(f"  Matrix density:        {len(train)/possible*100:.2f}%")

print("\n" + "="*60)
print("DIRECTION SPLIT AMONG MOVERS (for the DIR-AUROC framing)")
print("="*60)
mv = train[train["label"] != "none"]
up_share = (mv["label"] == "up").mean()
print(f"  Among movers: {up_share*100:.1f}% up, {(1-up_share)*100:.1f}% down")
print(f"  Always-up baseline accuracy on movers: {up_share:.3f}")

DATASET SIZE
Train rows: 7,705
Test rows:  1,813
Columns:    ['id', 'pert', 'gene', 'label']

LABEL DISTRIBUTION (the class balance — key slide fact)
  up   :  2,359  ( 30.6%)
  down :  1,086  ( 14.1%)
  none :  4,260  ( 55.3%)
  movers (up+down): 3,445 (44.7%)

UNIQUE ENTITIES
  Unique perts (train): 386
  Unique genes (train): 1,570
  Unique perts (test):  96
  Unique genes (test):  636

TRAIN/TEST DISJOINTNESS (why blocked validation is needed)
  Shared perts: 0  (0 = fully disjoint)
  Shared genes: 0  (0 = fully disjoint)

PAIR COVERAGE (how sparse the pert×gene matrix is)
  Observed pairs:        7,705
  Possible pert×gene:    606,020
  Matrix density:        1.27%

DIRECTION SPLIT AMONG MOVERS (for the DIR-AUROC framing)
  Among movers: 68.5% up, 31.5% down
  Always-up baseline accuracy on movers: 0.685
